# 13주 워크북 2 (강의자용 시연) — 추가 지식베이스와 음성 챗봇
학생용과 별도 파일. 예상 결과와 설명 포인트는 👩‍🏫 메모에 있다.

| 단계 | 예상 |
|---|---|
| 1 | 기존 5건 |
| 2~3 | 검증 지식 9건(기존 5 + 음성 4) |
| 4 | “빨강 보관” 질문: 전 A라인 조립(질문과 안 맞음) → 후 왼쪽 상자 |
| 5 | 답변 mp3 재생 |

## 0. 준비
`런타임 → 런타임 유형 변경 → T4 GPU`(없으면 CPU) 후 차례로 실행.

In [ ]:
# 0-1. 필요한 도구 설치 (약 30초, 처음 한 번만)
%pip install -q "gradio>=5,<7" gTTS qrcode

In [ ]:
# 0-2. 실습 데이터 내려받기: Google Drive → Colab (약 5초)
import os
import zipfile
import urllib.request

DATA_ID = '1KKtFDluctvGHcmrVbGn-eeDByb--dKHb'
BACKUP_URL = ('https://raw.githubusercontent.com/itlects/2026-1_ai-convergence-apply/main/week13/data/week13_data.zip')


def get_data():
    if os.path.isdir('week13_data'):
        return 'already'
    try:
        import gdown
        gdown.download(id=DATA_ID, output='week13_data.zip',
                       quiet=True)
        zipfile.ZipFile('week13_data.zip').extractall('.')
        return 'drive'
    except Exception as e:
        print('Drive 다운로드 실패 → GitHub 예비 주소 사용:', e)
        urllib.request.urlretrieve(BACKUP_URL, 'week13_data.zip')
        zipfile.ZipFile('week13_data.zip').extractall('.')
        return 'github'


print('받은 곳:', get_data())
for root, dirs, names in sorted(os.walk('week13_data')):
    if names:
        print(root, '→', len(names), '개 파일')

In [ ]:
# 음성 도구 함수: 변환(ffmpeg) · 읽기(wave) · 텍스트 변환(STT)
import subprocess
import wave
import numpy as np
from IPython.display import Audio, display


def to_wav16k(src, dst):
    """어떤 녹음 파일(m4a, mp3, webm, wav)이든 16kHz·모노·16bit WAV로 바꾼다."""
    subprocess.run(
        ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(src),
         '-ac', '1', '-ar', '16000', '-sample_fmt', 's16', str(dst)],
        check=True,
    )
    return str(dst)


def read_wav(path):
    """WAV 파일 → (샘플 배열 -1~1, 샘플링레이트)"""
    with wave.open(str(path), 'rb') as w:
        sr = w.getframerate()
        pcm = w.readframes(w.getnframes())
    x = np.frombuffer(pcm, dtype='<i2').astype(np.float32) / 32768
    return x, sr


def speech_to_text(path):
    """WAV → 한국어 텍스트"""
    x, sr = read_wav(path)
    out = stt({'raw': x, 'sampling_rate': sr},
              generate_kwargs={'language': 'korean',
                               'task': 'transcribe'})
    return out['text'].strip()


# STT가 자주 틀리는 현장 용어를 바로잡는 교정 사전 (틀린 말 → 바른 말)
FIX = {
    '바람 부품': '파랑 부품',
    '하랑 부품': '파랑 부품',
    '파란 부품': '파랑 부품',
    '빨간 부품': '빨강 부품',
    '부풍': '부품',
}


def fix_terms(text):
    for wrong, right in FIX.items():
        text = text.replace(wrong, right)
    return text

In [ ]:
# 음성인식(STT) 모델 불러오기: OpenAI Whisper-small
# 처음 실행 시 모델(약 1GB)을 내려받는다. GPU 약 30초, CPU 약 1~2분.
import torch
from transformers import pipeline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
stt = pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-small',
    device=DEVICE,
)
print('STT 준비 완료, 사용 장치:', DEVICE)

In [ ]:
# 텍스트 → 음성(TTS): Google TTS(gTTS)로 mp3 만들기
def text_to_speech(text, path='answer.mp3'):
    try:
        from gtts import gTTS
        gTTS(text, lang='ko').save(path)
        return path
    except Exception as e:
        print('TTS 실패(인터넷 확인):', e)
        return None

## 1. 기존 지식 불러오기
지난주 문서 RAG에서 만든 **문서 조각(청크)** 5건이다. 작업 매뉴얼에는 “어디에 보관하는지”가 없다.

In [ ]:
# 1-1. JSONL 읽기
import json
from pathlib import Path
import pandas as pd

DATA = Path('week13_data')
AUDIO = DATA / 'audio'


def load_jsonl(path):
    text = Path(path).read_text(encoding='utf-8')
    return [json.loads(line) for line in text.splitlines()
            if line.strip()]


def save_jsonl(rows, path):
    lines = [json.dumps(r, ensure_ascii=False) for r in rows]
    Path(path).write_text('\n'.join(lines) + '\n', encoding='utf-8')


base = load_jsonl(DATA / 'knowledge/base_knowledge.jsonl')
print('기존 지식', len(base), '건')
pd.DataFrame(base)[['id', 'label', 'text', 'source']]

## 2. 음성 → 추가 지식 만들기
추가 지식 생성 5단계: **수집(녹음) → 전사(STT) → 교정(용어 사전) → 구조화(JSON) → 검증(사람 확인)**

### 2-1. 워크북 1 결과 가져오기
워크북 1의 `field_knowledge.jsonl`이 있으면 `USE_MY_FILE = True`로 바꾸고 업로드한다. 없으면 예시 파일을 사용한다.

In [ ]:
# 2-1. 워크북 1 현장 지식
USE_MY_FILE = False   # 내 파일을 쓰려면 True

if USE_MY_FILE:
    from google.colab import files
    uploaded = files.upload()
    field = load_jsonl(list(uploaded)[0])
else:
    field = load_jsonl(DATA / 'knowledge/sample_field_knowledge.jsonl')
print('워크북 1 지식', len(field), '건')
pd.DataFrame(field)[['id', 'label', 'text', 'stt_raw', 'source']]

In [ ]:
# 2-2. 새 음성 메모 일괄 전사 → 지식 후보 (아직 검증 전)
new_files = ['voice_red_02.wav', 'voice_blue_02.wav']
candidates = []
for i, name in enumerate(new_files, 1):
    raw = speech_to_text(AUDIO / name)
    text = fix_terms(raw)
    label = '빨강' if '빨강' in text else (
        '파랑' if '파랑' in text else '공통')
    candidates.append({
        'id': f'voice-new-{i:02d}', 'label': label, 'text': text,
        'stt_raw': raw, 'source': name, 'type': 'voice_note',
        'verified': False,
    })
    print(name, '→', text)
    display(Audio(filename=str(AUDIO / name)))

### 2-3. 사람 검토 (Human-in-the-loop)
음성을 듣고 바른 문장을 `REVIEW`에 적은 것만 `verified=True`가 된다. **검증되지 않은 지식은 검색에 쓰지 않는다.**

In [ ]:
# 2-3. 사람 검토: {파일명: 들은 대로 고친 문장}
REVIEW = {
    'voice_red_02.wav': '빨강 부품에 균열이 보이면 재검사 상자로 옮깁니다.',
    'voice_blue_02.wav': '파랑 부품은 한 상자에 스무 개까지만 담습니다.',
}

for c in candidates:
    if REVIEW.get(c['source']):
        c['text'] = REVIEW[c['source']]
        c['verified'] = True
pd.DataFrame(candidates)[['source', 'text', 'verified']]

> 👩‍🏫 **강의자 메모** — 학생용은 voice_blue_02 검토가 비어 있어 처음엔 검증 지식이 8건, 학생이 문장을 채우면 9건이 된다. “검증되지 않은 자료는 검색 대상에서 제외”가 품질 관리의 핵심.

## 3. 품질 점검과 병합
- 빈 문장·검증 안 됨 → 제외
- **같은 문장**이 두 번 들어오면(띄어쓰기 무시) → 처음 것 하나만
- 결과를 `rag_knowledge.jsonl`로 저장 = **추가 지식베이스**

In [ ]:
# 3-1. 점검 · 중복 제거 · 저장
def build_kb(*groups):
    kb, seen = [], set()
    for rows in groups:
        for r in rows:
            if not r.get('verified') or not r.get('text', '').strip():
                print('제외(검증 안 됨/빈 문장):', r.get('source'))
                continue
            key = r['text'].replace(' ', '')
            if key in seen:
                print('제외(중복 문장):', r['source'])
                continue
            seen.add(key)
            kb.append(r)
    return kb


updated = build_kb(base, field, candidates)
save_jsonl(updated, 'rag_knowledge.jsonl')
print('기존', len(base), '건 → 추가 후', len(updated), '건')
pd.DataFrame(updated)[['id', 'label', 'text', 'source']]

## 4. 검색(Retrieval) 전/후 비교
**TF-IDF 글자 n-gram**: 문장을 2~3글자 조각으로 나눠 숫자 벡터로 만든다(조사 “은/는/을”이 붙어도 비슷하게 찾음).
**코사인 유사도**: 질문 벡터와 문서 벡터의 방향이 비슷할수록 1에 가깝다. `MIN_SCORE`보다 낮으면 “근거 없음”.

In [ ]:
# 4-1. 검색기
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

MIN_SCORE = 0.25


class Retriever:
    def __init__(self, rows):
        self.rows = rows
        docs = [r['label'] + ' ' + r['text'] for r in rows]
        self.vec = TfidfVectorizer(analyzer='char_wb',
                                   ngram_range=(2, 3))
        self.mat = self.vec.fit_transform(docs)

    def search(self, question, k=2):
        q = self.vec.transform([question])
        scores = cosine_similarity(q, self.mat)[0]
        order = scores.argsort()[::-1][:k]
        return [(float(scores[i]), self.rows[i]) for i in order
                if scores[i] >= MIN_SCORE]


before = Retriever(base)
after = Retriever(updated)
for s, r in after.search('빨강 부품은 어디에 보관해?'):
    print(f'{s:.2f}', r['text'], '|', r['source'])

In [ ]:
# 4-2. 같은 질문, 전/후 1순위 근거 비교
QUESTIONS = [
    '빨강 부품은 어디에 보관해?',
    '파랑 부품을 보관할 때 주의할 점은?',
    '균열이 보이면 어떻게 해?',
    '작업장 온도는 몇 도로 유지해?',
]


def top1(retriever, q):
    hits = retriever.search(q, k=1)
    if not hits:
        return '근거 없음', 0.0
    return hits[0][1]['text'], round(hits[0][0], 2)


rows = []
for q in QUESTIONS:
    b_text, b_s = top1(before, q)
    a_text, a_s = top1(after, q)
    rows.append([q, b_text, b_s, a_text, a_s])
pd.DataFrame(rows, columns=['질문', '전: 1순위 근거', '전 점수',
                            '후: 1순위 근거', '후 점수'])

> 👩‍🏫 **강의자 메모** — 예상: 빨강 보관 → 전 “A라인 조립”(관련 단어만 같고 답이 아님) / 후 “왼쪽 상자”. 파랑 주의 → 후 “습기·뚜껑”. 균열 → 후 “재검사 상자” 또는 매뉴얼 “불량 분류”. 온도 → 전·후 모두 근거 없음 = 지식에 없는 것은 답하지 않는다. 모델 재학습 없이 검색 결과가 바뀌었다는 점을 강조.

## 5. 음성 질문 → 검색 → 답변 → 음성(TTS)
마이크(음성 질문) → STT → 교정 → 검색 → 답변 문장 → TTS → 스피커

In [ ]:
# 5-1. 음성 챗봇 함수
def answer(question):
    hits = after.search(fix_terms(question), k=1)
    if not hits:
        return '관련 지식을 찾지 못했습니다. 현장 기록이 필요합니다.', None
    score, r = hits[0]
    return f"{r['text']} (출처: {r['source']})", r


def voice_chat(wav_path):
    q = speech_to_text(wav_path)
    text, hit = answer(q)
    print('🎙️ 질문(STT):', q)
    print('🔎 교정 질문 :', fix_terms(q))
    print('💬 답변     :', text)
    mp3 = text_to_speech(text.split(' (출처')[0])
    if mp3:
        display(Audio(filename=mp3))
    return text


for name in ['question_01.wav', 'question_02.wav']:
    display(Audio(filename=str(AUDIO / name)))
    voice_chat(AUDIO / name)
    print('-' * 40)

> 👩‍🏫 **강의자 메모** — 질문 음성도 STT 오류가 섞일 수 있지만 글자 조각 검색이라 핵심 단어(빨강·보관)가 맞으면 근거를 찾는다. 답변은 LLM이 새로 만든 문장이 아니라 검색된 근거를 그대로 읽는 구조(RAG의 R 단계)임을 설명.

## 6. 핸드폰 음성 챗봇
셀 실행 → QR 스캔 → 🎙️ 질문 녹음(또는 글자로 입력) → `질문하기` → 답변 음성 ▶ 재생.
질문 예: “빨강 부품은 어디에 보관해?”, “파랑 부품 주의할 점은?”, “작업장 온도는?”

In [ ]:
# 공유 주소를 QR 코드로 보여 주는 함수 (핸드폰 카메라로 스캔)
import qrcode
from IPython.display import Image as ShowImage


def show_qr(url):
    if not url:
        print('공유 주소가 없습니다. 위 출력의 오류를 확인하세요.')
        return
    qrcode.make(url).save('share_qr.png')
    print('핸드폰으로 접속:', url)
    display(ShowImage('share_qr.png', width=220))

In [ ]:
# 6-1. 핸드폰 음성 챗봇 (Gradio)
import gradio as gr


def ui_chat(audio_path, typed):
    if audio_path:
        wav = to_wav16k(audio_path, 'phone_question.wav')
        q = speech_to_text(wav)
    else:
        q = (typed or '').strip()
    if not q:
        return '', '질문을 녹음하거나 입력하세요.', None, []
    text, hit = answer(q)
    mp3 = text_to_speech(text.split(' (출처')[0])
    hits = [[round(s, 2), r['label'], r['text'], r['source']]
            for s, r in after.search(fix_terms(q), k=3)]
    return q, text, mp3, hits


with gr.Blocks(title='13주 음성 지식 챗봇') as demo:
    gr.Markdown('## 🔊 현장 지식 음성 챗봇')
    mic = gr.Audio(sources=['microphone', 'upload'], type='filepath',
                   label='🎙️ 질문 녹음')
    typed = gr.Textbox(label='또는 글자로 질문')
    ask = gr.Button('질문하기', variant='primary')
    q_out = gr.Textbox(label='인식된 질문(STT)')
    a_out = gr.Textbox(label='답변(근거 + 출처)', lines=2)
    speak = gr.Audio(label='🔊 답변 듣기', type='filepath')
    evid = gr.Dataframe(headers=['점수', '라벨', '근거', '출처'],
                        label='검색된 근거')
    ask.click(ui_chat, [mic, typed], [q_out, a_out, speak, evid])

demo.launch(share=True)
show_qr(demo.share_url)

## 7. 결과 저장
`rag_knowledge.jsonl`을 내려받아 결과보고 페이지에 첨부한다.

In [ ]:
# 7-1. 추가 지식베이스 내려받기
print('최종 지식베이스', len(updated), '건')
try:
    from google.colab import files
    files.download('rag_knowledge.jsonl')
except ImportError:
    print('Colab이 아닌 환경: 파일은 현재 폴더에 있습니다.')

### 👩‍🏫 정리 질문
1. 지식을 추가하면 Whisper나 색 분류 모델의 가중치가 바뀌나? → 아니오, 검색 대상(지식베이스)만 바뀜
2. STT 원문(stt_raw)을 함께 저장하는 이유? → 추적·재검토
3. “근거 없음”을 답하는 것이 왜 중요한가? → 지어낸 답(환각) 방지